In [1]:
import os
import json
import torch

from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0910 20:36:34.576000 32160 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [3]:
# ============================================================
# CELL 2 — LOAD PREPROCESSED DATA
# ============================================================

PREPROCESSED_DIR = "model2_preprocessed"


DATASET_PATH = os.path.join(
    PREPROCESSED_DIR,
    "dataset"
)

LABEL_MAP_PATH = os.path.join(
    PREPROCESSED_DIR,
    "label_mapping.json"
)


# Load tokenized dataset
dataset = load_from_disk(DATASET_PATH)


# Load label mapping
with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)


label2id = label_mapping["label2id"]

id2label = {
    int(k): v
    for k, v in label_mapping["id2label"].items()
}


print("Dataset loaded successfully!")
print(dataset)

print("\nNumber of examples:", len(dataset))
print("Number of labels:", len(label2id))

print("\nLabels:")
for idx, label in id2label.items():
    print(f"{idx}: {label}")

Dataset loaded successfully!
Dataset({
    features: ['id', 'text', 'label_name', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100
})

Number of examples: 100
Number of labels: 36

Labels:
0: RED_FLAG
1: associated_symptoms
2: breath_onset
3: breath_progression
4: breath_severity
5: cough_character
6: cough_frequency
7: cough_progression
8: diarrhea_frequency
9: dizziness_frequency
10: dizziness_onset
11: dizziness_progression
12: dizziness_triggers
13: dizziness_type
14: duration
15: fever_progression
16: fever_temperature
17: functional_impact
18: injury_or_trigger
19: nausea_onset
20: pain_location
21: pain_progression
22: pain_quality
23: pain_relieving_factors
24: pain_severity
25: pain_triggers
26: rash_character
27: rash_location
28: rash_progression
29: severity
30: throat_severity
31: urinary_frequency
32: urinary_onset
33: urinary_progression
34: vomiting
35: vomiting_frequency


In [4]:
# ============================================================
# CELL 3 — LOAD CLINICALBERT CLASSIFIER
# ============================================================

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

print("ClinicalBERT loaded successfully!")
print("Number of labels:", model.config.num_labels)

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\astha\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00

ClinicalBERT loaded successfully!
Number of labels: 36


In [5]:
# ============================================================
# CELL 4 — PREPARE DATASET FOR TRAINING
# ============================================================

# Keep only the columns needed by ClinicalBERT
columns_to_keep = [
    "input_ids",
    "token_type_ids",
    "attention_mask",
    "labels"
]

train_dataset = dataset.remove_columns(
    [
        column
        for column in dataset.column_names
        if column not in columns_to_keep
    ]
)

print("Training dataset prepared!")
print(train_dataset)

print("\nColumns:")
print(train_dataset.column_names)

print("\nNumber of examples:")
print(len(train_dataset))

Training dataset prepared!
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100
})

Columns:
['labels', 'input_ids', 'token_type_ids', 'attention_mask']

Number of examples:
100


In [7]:
# ============================================================
# CELL 5 — LOAD TOKENIZER + CREATE DATA COLLATOR
# ============================================================
from transformers import AutoTokenizer

TOKENIZER_PATH = os.path.join(
    PREPROCESSED_DIR,
    "tokenizer"
)

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH
)

print("Tokenizer loaded successfully!")


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created successfully!")

Tokenizer loaded successfully!
Data collator created successfully!


In [9]:
# ============================================================
# CELL 6 — TRAINING CONFIGURATION
# ============================================================

# Current Model 2 folder
BASE_DIR = os.getcwd()

# Where training checkpoints will be stored
OUTPUT_TRAINING_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "model2_clinicalbert"
)

# Where the final trained model will be saved
FINAL_MODEL_DIR = os.path.join(
    OUTPUT_TRAINING_DIR,
    "final"
)


training_args = TrainingArguments(
    output_dir=OUTPUT_TRAINING_DIR,

    # Training
    num_train_epochs=5,
    per_device_train_batch_size=8,

    # Learning
    learning_rate=2e-5,
    weight_decay=0.01,

    # Logging
    logging_steps=5,

    # Save checkpoint after each epoch
    save_strategy="epoch",

    # No external logging
    report_to="none",

    # Use FP16 if GPU is available
    fp16=torch.cuda.is_available()
)


print("Training configuration created!")
print("\nDevice:", "GPU" if torch.cuda.is_available() else "CPU")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)
print("Training output:", OUTPUT_TRAINING_DIR)
print("Final model:", FINAL_MODEL_DIR)

Training configuration created!

Device: GPU
Epochs: 5
Batch size: 8
Learning rate: 2e-05
Training output: c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_clinicalbert
Final model: c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_clinicalbert\final


In [10]:
# ============================================================
# CELL 7 — CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [11]:
# ============================================================
# CELL 8 — START TRAINING
# ============================================================

print("Starting Model 2 training...")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("Training completed successfully!")

Starting Model 2 training...


Step,Training Loss
5,3.656641
10,3.515186
15,3.242236
20,3.304004
25,3.045923
30,3.026172
35,2.926147
40,2.790479
45,2.949341
50,2.872168


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s]


Training completed successfully!


In [12]:
# ============================================================
# CELL 9 — SAVE FINAL TRAINED MODEL
# ============================================================

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Final Model 2 saved successfully!")
print()
print("Location:")
print(FINAL_MODEL_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.53it/s]

Final Model 2 saved successfully!

Location:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_clinicalbert\final
